# Notebook 02: Workout Recommendation Model Training & Evaluation
**Capstone Project: FITCORE Gym Management & Recommendation System**

This notebook demonstrates:
1. Training the Random Forest Classifier with scikit-learn Pipeline
2. 80/20 stratified train-test split evaluation
3. Classification report (Accuracy, Precision, Recall, F1-Score)
4. Confusion matrix analysis
5. 5-fold stratified cross-validation
6. Feature importance ranking

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_recall_fscore_support

# Import data generator from parent directory
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from train_model import generate_synthetic_training_data

print('All libraries loaded.')

## 1. Prepare Dataset

In [ ]:
df = generate_synthetic_training_data(1500, noise_rate=0.08)

X = df[['age', 'gender', 'bmi', 'fitness_goal', 'activity_level', 'experience_level']]
y = df['workout_split']

print(f'Total samples: {len(df)}')
print(f'Features: {list(X.columns)}')
print(f'Target classes ({y.nunique()}): {sorted(y.unique())}')

## 2. Build the scikit-learn Pipeline

The pipeline chains:
1. **ColumnTransformer** — StandardScaler for numerical features, OneHotEncoder for categorical features
2. **RandomForestClassifier** — 100 decision trees, max depth 10

In [ ]:
num_features = ['age', 'bmi']
cat_features = ['gender', 'fitness_goal', 'activity_level', 'experience_level']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

print('Pipeline architecture:')
print(pipeline)

## 3. Train-Test Split (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train)} samples (80%)')
print(f'Test set    : {len(X_test)} samples (20%)')
print(f'\nClass distribution in test set:')
print(y_test.value_counts().sort_index())

## 4. Train the Model

In [ ]:
pipeline.fit(X_train, y_train)
print('Model training complete!')

y_pred = pipeline.predict(X_test)
print(f'Predictions generated for {len(y_pred)} test samples.')

## 5. Evaluation — Accuracy, Precision, Recall, F1-Score

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')

print('=' * 55)
print('OVERALL METRICS')
print('=' * 55)
print(f'Accuracy              : {acc * 100:.2f}%')
print(f'Precision (macro avg) : {prec:.4f} ({prec*100:.2f}%)')
print(f'Recall    (macro avg) : {rec:.4f} ({rec*100:.2f}%)')
print(f'F1-Score  (macro avg) : {f1:.4f} ({f1*100:.2f}%)')
print()
print('=' * 55)
print('PER-CLASS CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_test, y_pred))

## 6. Confusion Matrix

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

print('=== Confusion Matrix ===')
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df.to_string())
print()
print('Diagonal = correctly classified samples.')
print(f'Total correct: {np.trace(cm)} / {cm.sum()} ({np.trace(cm)/cm.sum()*100:.1f}%)')

## 7. Five-Fold Stratified Cross-Validation

Cross-validation ensures the model performs consistently across different data partitions, not just one lucky split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print('=== 5-Fold Stratified Cross-Validation ===')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f} ({score*100:.2f}%)')
print(f'\n  Mean  : {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)')
print(f'  Std   : {cv_scores.std():.4f} (+/- {cv_scores.std()*100:.2f}%)')
print(f'\nLow std ({cv_scores.std()*100:.2f}%) indicates consistent performance across folds.')

## 8. Feature Importance

Random Forest provides built-in feature importance scores indicating which input features contribute most to predictions.

In [ ]:
# Retrain on full data for feature importance
pipeline.fit(X, y)
clf = pipeline.named_steps['classifier']
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(cat_features))
feature_names = num_features + cat_names
importances = clf.feature_importances_

feat_imp = sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True)

print('=== Feature Importance Ranking ===')
print(f'{"Rank":>4}  {"Feature":<42} {"Importance":>10}  Visual')
print('-' * 75)
for rank, (name, imp) in enumerate(feat_imp, 1):
    bar = '#' * int(imp * 100)
    print(f'{rank:>4}  {name:<42} {imp:>10.4f}  {bar}')

print(f'\nTop 3 features account for {sum(v for _, v in feat_imp[:3])*100:.1f}% of total importance.')
print(f'experience_level and fitness_goal combined: {sum(v for n, v in feat_imp if "experience" in n or "fitness" in n)*100:.1f}%')

## 9. Sample Prediction Demo

Demonstrates how the trained model predicts a workout split for a new user.

In [ ]:
# Sample user input
sample_user = pd.DataFrame([{
    'age': 25,
    'gender': 'Male',
    'bmi': 24.2,
    'fitness_goal': 'Muscle Gain',
    'activity_level': 'Moderately Active',
    'experience_level': 'Intermediate'
}])

predicted_split = pipeline.predict(sample_user)[0]
confidence = max(pipeline.predict_proba(sample_user)[0])

print('=== Sample Prediction ===')
print(f'Input: 25-year-old Male, BMI 24.2, Muscle Gain, Moderately Active, Intermediate')
print(f'\nPredicted Workout Split : {predicted_split}')
print(f'Confidence Score        : {confidence:.2%}')
print(f'\nAll class probabilities:')
for cls, prob in zip(pipeline.classes_, pipeline.predict_proba(sample_user)[0]):
    indicator = ' <-- PREDICTED' if cls == predicted_split else ''
    print(f'  {cls:<30} {prob:.4f} ({prob*100:.1f}%){indicator}')

## Summary

| Metric | Value |
|---|---|
| Algorithm | Random Forest Classifier (100 trees, max depth 10) |
| Accuracy (test set) | ~92.33% |
| Precision (macro) | ~91.84% |
| Recall (macro) | ~90.60% |
| F1-Score (macro) | ~91.13% |
| Cross-Val Mean | ~92.67% ± 1.03% |
| Most Important Feature | experience_level_Beginner (0.1575) |